In [27]:
from google.cloud import aiplatform
import os
import time
import re

# === CONFIGURACIÓN ===
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../../../bubbo-dfba0-47e395cdcdc7.json"
PROJECT = "bubbo-dfba0"
LOCATION = "europe-southwest1"
STAGING_BUCKET = "gs://embeddings_new_bucket"
INDEX_NAME = "alpha_recs_movies_tv_tree_ah_eu_sw1"
EMBEDDINGS_URI = "gs://embeddings_new_bucket/embeddings/index_data/"
DIMENSIONS = 768
ENDPOINT_ID = "projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272"




In [28]:
# === INICIAR ===
aiplatform.init(project=PROJECT, location=LOCATION, staging_bucket=STAGING_BUCKET)

# === ELIMINAR ÍNDICE ANTERIOR SI EXISTE ===
existing_indexes = aiplatform.MatchingEngineIndex.list(filter=f'display_name="{INDEX_NAME}"')
for idx in existing_indexes:
    print(f"Eliminando índice previo: {idx.display_name}")
    idx.delete()
    idx.wait()

# === CREAR ÍNDICE  ===
try:
    print(f"Creando nuevo índice: {INDEX_NAME}")
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name=INDEX_NAME,
        contents_delta_uri=EMBEDDINGS_URI,
        description="Matching Engine Index",
        dimensions=DIMENSIONS,
        approximate_neighbors_count=250,
        leaf_node_embedding_count=1000,
        distance_measure_type=aiplatform.matching_engine.matching_engine_index_config.DistanceMeasureType.COSINE_DISTANCE,
        index_update_method="STREAM_UPDATE"
    )
    index.wait()  # Esperar a que el índice se cree
    print(f"Índice {INDEX_NAME} creado correctamente.")
except Exception as e:
    print(f"Error al crear el índice: {e}")
    raise



# === DESPLEGAR AL ENDPOINT ===
try:
    endpoint = aiplatform.MatchingEngineIndexEndpoint(index_endpoint_name=ENDPOINT_ID)
    deployed_index_id = re.sub(r'[^a-zA-Z0-9_]', '_', INDEX_NAME)[:63]
    if not deployed_index_id[0].isalpha():
        deployed_index_id = f"a_{deployed_index_id}"

    # Eliminar despliegue previo con mismo ID
    for d in endpoint.deployed_indexes:
        if d.id == deployed_index_id:
            print(f"Desplegando índice previo con el mismo ID: {deployed_index_id}")
            endpoint.undeploy_index(deployed_index_id=d.id)
            time.sleep(5)

    # Desplegar el nuevo índice
    print(f"Desplegando índice al endpoint: {deployed_index_id}")
    endpoint.deploy_index(
        index=index,
        deployed_index_id=deployed_index_id,
        display_name="deployed-alpha-index"
    )

    print("✅ Índice creado, cargado y desplegado correctamente.")
except Exception as e:
    print(f"Error al desplegar el índice: {e}")
    raise


Creando nuevo índice: alpha_recs_movies_tv_tree_ah_eu_sw1
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/75629471929/locations/europe-southwest1/indexes/5403158468565663744/operations/2265949428823097344
MatchingEngineIndex created. Resource name: projects/75629471929/locations/europe-southwest1/indexes/5403158468565663744
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/75629471929/locations/europe-southwest1/indexes/5403158468565663744')
Índice alpha_recs_movies_tv_tree_ah_eu_sw1 creado correctamente.
Desplegando índice al endpoint: alpha_recs_movies_tv_tree_ah_eu_sw1
Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272/operations/8697089696708165632
MatchingEngi

In [1]:
!
1
1
from google.cloud import storage
import os

# Establecer la variable de entorno para las credenciales
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../../../bubbo-dfba0-47e395cdcdc7.json"

# Crear cliente de almacenamiento utilizando el archivo de credenciales
client = storage.Client.from_service_account_json("../../../bubbo-dfba0-47e395cdcdc7.json")

# Nombre de tu bucket y archivos
bucket_name = "embeddings_new_bucket"
original_file = "embeddings/index_data/all_embeddings.jsonl"
new_file = "embeddings/index_data/all_embeddings.json"

# Obtener el bucket
bucket = client.get_bucket(bucket_name)

# Crear los blobs (archivos) para el archivo original y el nuevo
blob_original = bucket.blob(original_file)

# Copiar el archivo original al nuevo archivo (renombrarlo)
bucket.copy_blob(blob_original, bucket, new_file)

# Eliminar el archivo original (si lo deseas)
blob_original.delete()

print(f"Archivo renombrado de {original_file} a {new_file}")


NotFound: 404 POST https://storage.googleapis.com/storage/v1/b/embeddings_new_bucket/o/embeddings%2Findex_data%2Fall_embeddings.jsonl/copyTo/b/embeddings_new_bucket/o/embeddings%2Findex_data%2Fall_embeddings.json?prettyPrint=false: No such object: embeddings_new_bucket/embeddings/index_data/all_embeddings.jsonl

In [37]:
from google.cloud import aiplatform_v1
import numpy as np

# === CONFIGURACIÓN ===
API_ENDPOINT = "961152428.europe-southwest1-75629471929.vdb.vertexai.goog"
INDEX_ENDPOINT = "projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272"
DEPLOYED_INDEX_ID = "alpha_recs_movies_tv_tree_ah_eu_sw1"
VECTOR_DIMENSIONS = 768

# === CLIENTE DE VECTOR SEARCH ===
client_options = {
    "api_endpoint": API_ENDPOINT
}
vector_search_client = aiplatform_v1.MatchServiceClient(client_options=client_options)

# === VECTOR DE CONSULTA ALEATORIO ===
random_vector = np.random.rand(VECTOR_DIMENSIONS).astype(float).tolist()

# === CONSTRUIR CONSULTA ===
datapoint = aiplatform_v1.IndexDatapoint(
    feature_vector=random_vector
)

query = aiplatform_v1.FindNeighborsRequest.Query(
    datapoint=datapoint,
    neighbor_count=10
)

request = aiplatform_v1.FindNeighborsRequest(
    index_endpoint=INDEX_ENDPOINT,
    deployed_index_id=DEPLOYED_INDEX_ID,
    queries=[query],
    return_full_datapoint=False
)

# === EJECUTAR CONSULTA ===
response = vector_search_client.find_neighbors(request=request)

# === MOSTRAR RESULTADOS ===
print("\n🔍 Resultados KNN:")
for neighbor in response.nearest_neighbors[0].neighbors:
    print(f"ID: {neighbor.datapoint.datapoint_id}, Distancia: {neighbor.distance:.4f}")



🔍 Resultados KNN:
ID: 622929, Distancia: 0.8536
ID: 196603, Distancia: 0.8651
ID: 249170, Distancia: 0.8687
ID: 5440930000, Distancia: 0.8687
ID: 120183, Distancia: 0.8698
ID: 590513, Distancia: 0.8730
ID: 314334, Distancia: 0.8738
ID: 297607, Distancia: 0.8745
ID: 466578, Distancia: 0.8751
ID: 34464, Distancia: 0.8763
